# Huấn luyện Auxiliary Density Head cho YOLO-World (P3 - Stride 8)
Notebook train nhánh **Density Head** trên tầng P3 (stride 8, 80×80) — độ phân giải không gian tốt hơn P4 cũ.

**Lưu ý:** Đảm bảo bạn đã Mount bộ dữ liệu `FSC-147` vào đường dẫn `/kaggle/input/fsc147`.

In [1]:
!pip install -q ultralytics opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.5 MB/s eta 0:00:00


## 1. Import Thư Viện & Cấu Hình

In [2]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF
import numpy as np
import cv2
from typing import List, Tuple, Dict, Optional, Any
from ultralytics import YOLOWorld

# ==========================================
# CONFIGURATION & KAGGLE PATHS
# ==========================================
DATA_DIR: str = "/kaggle/input/datasets/xuncngng/fsc147-0/FSC147"  # Cập nhật theo tên dataset trên Kaggle của bạn
IMAGE_DIR: str = os.path.join(DATA_DIR, "images_384_VarV2")
SPLIT_PATH: str = os.path.join(DATA_DIR, "Train_Test_Val_FSC_147.json")
ANNO_PATH: str = os.path.join(DATA_DIR, "annotation_FSC147_384.json")
CHECKPOINT_DIR: str = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE: int = 8
EPOCHS: int = 50
LEARNING_RATE: float = 1e-4
DEVICE: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE: int = 640
FEATURE_STRIDE: int = 8   # P3 (stride 8) -> density map 80x80 (tốt hơn P4 40x40)

print(f"Sử dụng thiết bị: {DEVICE}")
print(f"Density Map size: {IMAGE_SIZE // FEATURE_STRIDE}x{IMAGE_SIZE // FEATURE_STRIDE}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Sử dụng thiết bị: cuda
Density Map size: 80x80


## 2. Dataset & Target Generation

In [3]:
def generate_density_map(image_shape: Tuple[int, int], points: List[List[float]], sigma: float = 4.0) -> np.ndarray:
    """
    Sinh ra Heatmap Mật độ từ tập hợp các điểm Ground Truth.

    Args:
        image_shape (Tuple[int, int]): Kích thước heatmap (H, W)
        points (List[List[float]]): Danh sách tọa độ [x, y]
        sigma (float): Độ lan tỏa của Gaussian blur

    Returns:
        np.ndarray: Density map (H, W), sum ≈ số lượng vật thể
    """
    H, W = image_shape
    density_map: np.ndarray = np.zeros((H, W), dtype=np.float32)
    if not points:
        return density_map
    for point in points:
        x, y = min(int(point[0]), W - 1), min(int(point[1]), H - 1)
        density_map[y, x] = 1.0
    density_map = cv2.GaussianBlur(density_map, (15, 15), sigma)
    total_sum = density_map.sum()
    if total_sum > 0:
        density_map = (density_map / total_sum) * len(points)
    return density_map


class FSC147Dataset(Dataset):
    def __init__(self, split: str = "train") -> None:
        super().__init__()
        if not os.path.exists(SPLIT_PATH):
            raise FileNotFoundError(f"Không tìm thấy {SPLIT_PATH}. Hãy kiểm tra lại DATA_DIR.")
        with open(SPLIT_PATH, "r") as f:
            self.images: List[str] = json.load(f).get(split, [])
        with open(ANNO_PATH, "r") as f:
            self.annos: Dict[str, Any] = json.load(f)

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        img_name = self.images[idx]
        img_path = os.path.join(IMAGE_DIR, img_name)

        image = Image.open(img_path).convert("RGB")
        W, H = image.size
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
        img_tensor: torch.Tensor = TF.to_tensor(image)  # (3, 640, 640)

        gt_points: List[List[float]] = self.annos.get(img_name, {}).get("points", [])
        scaled_points: List[List[float]] = [
            [p[0] * IMAGE_SIZE / W, p[1] * IMAGE_SIZE / H] for p in gt_points
        ]

        # P3 stride=8 -> density map size = 640/8 = 80x80
        feat_size: int = IMAGE_SIZE // FEATURE_STRIDE  # 80
        feat_points = [[p[0] / FEATURE_STRIDE, p[1] / FEATURE_STRIDE] for p in scaled_points]
        density_map: np.ndarray = generate_density_map((feat_size, feat_size), feat_points)
        density_tensor: torch.Tensor = torch.from_numpy(density_map).unsqueeze(0)  # (1, 80, 80)

        return img_tensor, density_tensor

## 3. Kiến Trúc Mô Hình (P3 - 128 channels)

In [4]:
class DensityHead(nn.Module):
    """
    Mạng CNN Auxiliary nội suy Heatmap Mật Độ từ Feature Map.
    Input: P3 feature (128 channels, 80x80)
    Output: Density Map (1 channel, 80x80)
    """
    def __init__(self, in_channels: int) -> None:
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1)
        )
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.nn.functional.relu(self.conv_layers(x))


class DensityYOLOWorld(nn.Module):
    """
    Wrapper YOLO-World với Auxiliary Density Head gắn trên P3 (stride 8, 80x80).
    YOLOv8s: P3=128ch, P4=256ch, P5=512ch.
    """
    def __init__(self, yolo_weights: str = "yolov8s-world.pt") -> None:
        super().__init__()
        self.yolo = YOLOWorld(yolo_weights)
        for param in self.yolo.parameters():
            param.requires_grad = False
        self.features: Optional[torch.Tensor] = None
        self._register_hook()
        # P3 có 128 channels
        self.density_head = DensityHead(in_channels=128)

    def _register_hook(self) -> None:
        def hook_fn(module: nn.Module, input_args: tuple) -> None:
            feats = input_args[0]
            if isinstance(feats, (list, tuple)) and len(feats) >= 3:
                self.features = feats[0]  # P3 (stride 8, 80x80)
            else:
                self.features = feats
        self.yolo.model.model[-1].register_forward_pre_hook(hook_fn)

    def forward(self, x: torch.Tensor) -> Tuple[Any, torch.Tensor]:
        raw_preds = self.yolo.model(x)
        if self.features is None:
            raise RuntimeError("Hook không bắt được Feature Map.")
        return raw_preds, self.density_head(self.features)

    def train_mode(self) -> None:
        self.yolo.eval()
        self.density_head.train()

    def eval_mode(self) -> None:
        self.yolo.eval()
        self.density_head.eval()

## 4. Vòng Lặp Huấn Luyện

In [5]:
def train() -> None:
    print("Khởi tạo mô hình Hybrid Density YOLO-World (P3)...")
    model = DensityYOLOWorld("yolov8s-world.pt").to(DEVICE)
    model.train_mode()

    print("Chuẩn bị dữ liệu...")
    try:
        train_dataset = FSC147Dataset(split="train")
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    except Exception as e:
        print(f"LỖI tải dữ liệu: {e}")
        return

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.density_head.parameters(), lr=LEARNING_RATE)

    print(f"Bắt đầu Training {EPOCHS} epochs ({len(train_dataset)} ảnh)...")

    for epoch in range(EPOCHS):
        epoch_loss: float = 0.0
        model.density_head.train()

        for batch_idx, (images, gt_density) in enumerate(train_loader):
            images = images.to(DEVICE)
            gt_density = gt_density.to(DEVICE)

            optimizer.zero_grad()
            _, pred_density = model(images)
            loss = criterion(pred_density, gt_density)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if (batch_idx + 1) % 50 == 0:
                print(f"  Epoch [{epoch+1}/{EPOCHS}] | Batch [{batch_idx+1}/{len(train_loader)}] | Loss: {loss.item():.6f}")

        avg_loss: float = epoch_loss / len(train_loader)
        print(f"=> Epoch {epoch+1}/{EPOCHS} | Average Loss: {avg_loss:.6f}")

    save_path = os.path.join(CHECKPOINT_DIR, "density_head_best.pth")
    torch.save(model.density_head.state_dict(), save_path)
    print(f"Đã lưu checkpoint tại: {save_path}")

In [6]:
# Chạy huấn luyện
train()

Khởi tạo mô hình Hybrid Density YOLO-World (P3)...
Chuẩn bị dữ liệu...
Bắt đầu Training 50 epochs (3659 ảnh)...
  Epoch [1/50] | Batch [50/458] | Loss: 0.014707
  Epoch [1/50] | Batch [100/458] | Loss: 0.000104
  Epoch [1/50] | Batch [150/458] | Loss: 0.000136
  Epoch [1/50] | Batch [200/458] | Loss: 0.000048
  Epoch [1/50] | Batch [250/458] | Loss: 0.000061
  Epoch [1/50] | Batch [300/458] | Loss: 0.000065
  Epoch [1/50] | Batch [350/458] | Loss: 0.000169
  Epoch [1/50] | Batch [400/458] | Loss: 0.000070
  Epoch [1/50] | Batch [450/458] | Loss: 0.000233
=> Epoch 1/50 | Average Loss: 0.000283
  Epoch [2/50] | Batch [50/458] | Loss: 0.000219
  Epoch [2/50] | Batch [100/458] | Loss: 0.000126
  Epoch [2/50] | Batch [150/458] | Loss: 0.000053
  Epoch [2/50] | Batch [200/458] | Loss: 0.003493
  Epoch [2/50] | Batch [250/458] | Loss: 0.000105
  Epoch [2/50] | Batch [300/458] | Loss: 0.000191
  Epoch [2/50] | Batch [350/458] | Loss: 0.000112
  Epoch [2/50] | Batch [400/458] | Loss: 0.000054
 